# 沖縄県用Jageocoder辞書 作成ツール（build_okinawa_jageocoder）

`jageocoder-converter` を使って **沖縄県（都道府県コード47）のみ** のJageocoderローカル辞書を作成し、
ZIPファイルとしてダウンロードするための補助Notebookです。

**このNotebookは`sheltermatch.ipynb`本体ではありません。**
`sheltermatch.ipynb` は住所→座標変換（ジオコーディング）にJageocoderのローカル辞書を利用しますが、
その辞書を作る処理はここに分離しています。

**両Notebookの関係**

```
build_okinawa_jageocoder.ipynb で辞書を一度作成
↓
okinawa_jageocoder.zip を保存
↓
sheltermatch.ipynb で保存してある okinawa_jageocoder.zip をアップロード
↓
住所→座標変換に利用
```

Google Driveは使用しません。辞書はColabランタイム上の一時ディレクトリに作成し、完成後にZIP化して
ブラウザへダウンロードします（Colabのランタイムが終了するとこの一時ディレクトリ自体は消えるため、
必ずZIPを保存してください）。

**重要: このNotebookは毎回実行するものではありません。**
辞書を新規作成する時、または更新したい時だけ実行してください。全国版に比べれば小規模ですが、
辞書生成には数分〜数十分かかることがあります。既に辞書がある場合は、既定では上書きしません
（詳細は「3. 出力先設定」を参照）。

## 2. 必要ライブラリのインストール

Google Colabに標準で入っていない `jageocoder` と `jageocoder-converter` をインストールします。

In [ ]:
%pip install -q jageocoder jageocoder-converter

## 3. 出力先設定

このNotebookで編集が必要な設定はこのセルだけです。

In [ ]:
# ===== 設定（このセルの値を必要に応じて編集してください） =====

# 生成する辞書の対象都道府県コード（47 = 沖縄県）
PREF_CODE = "47"

# 辞書の出力先ディレクトリ（Colabランタイム上の一時ディレクトリ）。
# このディレクトリはColabのランタイムが終了すると消えるため、生成後は必ず
# 「6. 辞書のZIP化とダウンロード」でZIPファイルとしてダウンロード・保存してください。
JAGEOCODER_DB_DIR = "/content/okinawa_db"

# 出力先に既に辞書がある場合に、上書きして再作成するかどうか。
# False（既定）の場合、既存の辞書があれば辞書生成をスキップし、上書きしません。
# 再作成したい場合のみ True に変更してください。
FORCE_REBUILD = False

# jageocoder-converterは辞書生成時に、元データの利用規約への同意を対話的に(input()で)
# 求めます。ColabのIPython(!)実行は対話的な入力を受け付けられないため、辞書の新規作成・
# 再作成をそのまま試みると、確認表示のまま処理が進まなくなる可能性があります。
# そのため、このNotebookでは辞書を新規作成・再作成する前に、必ず下記のドキュメントで
# 利用規約の内容をご自身で確認したうえで、このセルの ACCEPT_TERMS を True に変更することを
# 必須にしています（True の場合のみ、確認済みとして --quiet を付けて生成します）。
# 参考: https://github.com/t-sagara/jageocoder-converter
# False（既定）のままでは、辞書の新規作成・再作成は行われません
# （既存の辞書をそのまま使う場合は、この設定は関係ありません）。
ACCEPT_TERMS = False

print("設定を読み込みました。")
print(f"  PREF_CODE         = '{PREF_CODE}'")
print(f"  JAGEOCODER_DB_DIR = '{JAGEOCODER_DB_DIR}'")
print(f"  FORCE_REBUILD     = {FORCE_REBUILD}")
print(f"  ACCEPT_TERMS      = {ACCEPT_TERMS}")

## 4. 沖縄県辞書の生成

`jageocoder-converter` を使い、沖縄県（都道府県コード47）のみを対象に辞書を生成します。
街区・地番レベル以上の精度を確保するため `--no-gaiku` は指定しません。また、通常の住所照合精度を
優先し、住居表示住所データも省略せず、都道府県コード（`47`）のみを指定するシンプルな構成にしています。

処理の流れは次の通りです。

1. 既に出力先に辞書があり `FORCE_REBUILD = False`（既定）の場合は、生成を行わずそのまま使います。
2. 新規作成する場合、または `FORCE_REBUILD = True` の場合は、まず `ACCEPT_TERMS` を確認します。
   `ACCEPT_TERMS = False`（既定）のままでは、Colab上で利用規約への同意確認が入力できず
   停止する可能性があるため、**生成処理自体を開始せずエラーで停止します**（既存の辞書がある
   場合も削除しません）。「3. 出力先設定」に記載のドキュメントで利用規約を確認したうえで
   `ACCEPT_TERMS = True` に変更し、このセルを再実行してください。
3. `ACCEPT_TERMS = True` の場合のみ、（`FORCE_REBUILD = True` で既存辞書があれば削除した上で）
   確認済みとして `--quiet` を付けて生成コマンドを実行します。

生成コマンドが途中で失敗しても、このセル自体は次に進んでしまうため、コマンドの終了コードを
明示的に確認します。ここで異常終了と判定された場合、次の「5. 生成結果の確認」以降は
成功として扱いません。

In [ ]:
import os
import shutil

os.makedirs(JAGEOCODER_DB_DIR, exist_ok=True)
existing_files = os.listdir(JAGEOCODER_DB_DIR)

# このセルでの生成コマンドを実行したかどうかにかかわらず、後続セルが成功判定に使うフラグ。
# 生成をスキップした場合（既存辞書をそのまま使う場合）はTrueのままにする。
generation_ok = True

if existing_files and not FORCE_REBUILD:
    print("既存の辞書が見つかりました。意図しない上書きを防ぐため、辞書生成をスキップします。")
    print(f"  辞書ディレクトリ: {JAGEOCODER_DB_DIR}")
    print(f"  既存ファイル/フォルダ数: {len(existing_files)}件")
    print("再作成する場合は、上の「3. 出力先設定」で FORCE_REBUILD = True に変更してから、このセルを再実行してください。")
elif not ACCEPT_TERMS:
    generation_ok = False
    raise RuntimeError(
        "辞書を新規作成・再作成するには、jageocoder-converterが使用するデータの利用規約を"
        "ご自身で確認したうえで、上の「3. 出力先設定」で ACCEPT_TERMS = True に変更してから、"
        "このセルを再実行してください。既存の辞書は削除していません。"
    )
else:
    if existing_files and FORCE_REBUILD:
        print("FORCE_REBUILD=True のため、既存の辞書を削除してから再作成します。")
        for name in existing_files:
            target_path = os.path.join(JAGEOCODER_DB_DIR, name)
            if os.path.isdir(target_path):
                shutil.rmtree(target_path)
            else:
                os.remove(target_path)
        os.makedirs(JAGEOCODER_DB_DIR, exist_ok=True)

    print(f"沖縄県（都道府県コード {PREF_CODE}）のJageocoder辞書生成を開始します。")
    print("データ量によっては数分〜数十分かかることがあります。")
    print("ACCEPT_TERMS=True のため、利用規約に同意済みとして --quiet を付けて実行します。")

    !python -m jageocoder_converter convert --db-dir="{JAGEOCODER_DB_DIR}" --quiet {PREF_CODE}

    # IPython(Colab)の!実行は、コマンドが異常終了してもNotebookの処理は次に進んでしまうため、
    # 実行直後にIPythonが設定する終了コード(_exit_code)を明示的に確認する。
    # _exit_code が取得できない場合も含め、0以外・未定義は失敗として扱う（成功を誤判定しないため）。
    exit_code = globals().get("_exit_code", 1)
    if exit_code != 0:
        generation_ok = False
        print(f"辞書生成コマンドが異常終了しました（終了コード: {exit_code}）。上記の出力を確認してください。")
    else:
        print("辞書生成コマンドが正常終了しました。")

## 5. 生成結果の確認

生成コマンドが正常終了したことを前提に、辞書ファイルが出力先に作成されたか、
またJageocoderから実際に読み込めるかを確認します。ここで両方が確認できた場合のみ、
`dictionary_ready = True` とし、次の「6. 辞書のZIP化とダウンロード」に使います。

In [ ]:
import jageocoder

dictionary_ready = False
generated_files = os.listdir(JAGEOCODER_DB_DIR)

if not generation_ok:
    print("辞書生成コマンドが異常終了しているため、確認をスキップします。上のセルの出力を確認してください。")
elif not generated_files:
    print("辞書ディレクトリが空です。生成に失敗している可能性があります。上のセルの出力を確認してください。")
else:
    print(f"辞書ディレクトリにファイルが生成されています（{len(generated_files)}件）。")
    try:
        jageocoder.init(db_dir=JAGEOCODER_DB_DIR)
        print("Jageocoderで辞書を正常に読み込めることを確認しました。")
        dictionary_ready = True
    except Exception as error:
        print("辞書ファイルは生成されていますが、Jageocoderからの読み込みに失敗しました。")
        print(f"詳細: {error}")

## 6. 辞書のZIP化とダウンロード

前のセルで辞書がJageocoderから実際に読み込めることを確認できた場合のみ、辞書ディレクトリを
ZIP化してブラウザへダウンロードします。ZIP内には辞書ディレクトリの中身のみを格納し、余分な
親ディレクトリは含めません（展開したディレクトリをそのまま `jageocoder.init(db_dir=...)` に
渡せる構造にします）。

ダウンロードした `okinawa_jageocoder.zip` は、`sheltermatch.ipynb` の `ENABLE_GEOCODING = True`
時のアップロード先として使ってください。

In [ ]:
import zipfile

from google.colab import files

ZIP_FILENAME = "okinawa_jageocoder.zip"

if not dictionary_ready:
    print("辞書が正常に利用できる状態になっていないため、ZIP化・ダウンロードは行いません。")
    print("「4. 沖縄県辞書の生成」「5. 生成結果の確認」の出力を確認してください。")
else:
    # 辞書ディレクトリの中身をそのままZIPのトップレベルに格納する（余分な親ディレクトリを含めない）。
    with zipfile.ZipFile(ZIP_FILENAME, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for root, _dirs, filenames in os.walk(JAGEOCODER_DB_DIR):
            for filename in filenames:
                file_path = os.path.join(root, filename)
                arcname = os.path.relpath(file_path, JAGEOCODER_DB_DIR)
                zip_file.write(file_path, arcname)

    print(f"辞書ディレクトリを '{ZIP_FILENAME}' にZIP化しました。")
    print("sheltermatch.ipynb でこのZIPファイルをアップロードして利用してください。")

    files.download(ZIP_FILENAME)